# A2 experiments

This notebook reproduces Experiment A2 (corresponding to Figure 3 from the paper) from the released files. A fresh run writes only to `reproduced/`; it does not modify `data/` or `figures/`.

A2 uses 50 structured $100\times20$ matrices, $A=\begin{bmatrix}I\_{20}\\[1mm] \frac{1}{4}\widetilde{G}\end{bmatrix},$ where the 80 rows of $\widetilde{G}$ are normalized Gaussian directions. Each matrix is run at the same 15 values of $p$ as A1.


In [ ]:
from pathlib import Path
import platform
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
FIG_DIR = ROOT / "figures"
REPRO_DIR = ROOT / "reproduced"
REPRO_DIR.mkdir(exist_ok=True)

A1_PRIMARY_PATH = ROOT.parent / "A1" / "data" / "A1_primary_summary.csv"

M = 100
N = 20
N_ANCHOR = 20
N_CLOUD = 80
BETA = 1 / 4
DEV_SEED = 3000
FINAL_SEEDS = np.arange(3001, 3051, dtype=int)
P_GRID = np.array([
    3, 4, 5.7, 7.3, 10.8, 14.2, 19.6, 27.9,
    32, 38.4, 54.1, 64, 74.8, 98.6, 127.3,
], dtype=float)
THRESHOLDS = (1e-4, 1e-6, 1e-8)
PRIMARY_TOL = 1e-8
MAX_ITER = 10_000

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("A2 runs:", len(FINAL_SEEDS) * len(P_GRID))


## 1. Matrix generator and released bank

The released matrix bank is retained in `data/`. We regenerate the matrices independently from the recorded seeds and compare them with that bank.


In [ ]:
def make_a2_matrix(seed):
    rng = np.random.default_rng(int(seed))
    G = rng.standard_normal((N_CLOUD, N))
    G /= np.linalg.norm(G, axis=1, keepdims=True)
    return np.vstack([np.eye(N), BETA * G])

released_bank_file = np.load(DATA_DIR / "A2_matrix_bank.npz")
released_matrices = released_bank_file["matrices"]
released_seeds = released_bank_file["seeds"].astype(int)

regenerated_matrices = np.stack([make_a2_matrix(seed) for seed in FINAL_SEEDS])

assert released_matrices.shape == (50, M, N)
assert np.array_equal(released_seeds, FINAL_SEEDS)

bank_exact = np.array_equal(regenerated_matrices, released_matrices)
max_bank_diff = np.max(np.abs(regenerated_matrices - released_matrices))

if not bank_exact:
    assert np.allclose(regenerated_matrices, released_matrices, rtol=1e-14, atol=1e-14)

np.savez_compressed(
    REPRO_DIR / "A2_matrix_bank.npz",
    matrices=regenerated_matrices,
    seeds=FINAL_SEEDS,
    beta=np.asarray(BETA),
    m=np.asarray(M),
    n=np.asarray(N),
)

print("Regenerated matrix bank matches released bank exactly:", bank_exact)
print("Maximum absolute matrix-entry difference:", max_bank_diff)


## 2. Geometry diagnostics

The protocol planned a ratio of mean anchor leverage to mean cloud leverage. The implementation used the ratio of the two group medians, which is the statistic reported in the paper. We recompute both here. This diagnostic does not enter the Lewis iteration.


In [ ]:
def leverage_scores_qr(A):
    Q, _ = np.linalg.qr(A, mode="reduced")
    return np.sum(Q * Q, axis=1)

geom_rows = []
for seed, A in zip(FINAL_SEEDS, regenerated_matrices):
    row_norms = np.linalg.norm(A, axis=1)
    lev = leverage_scores_qr(A)
    anchor = lev[:N_ANCHOR]
    cloud = lev[N_ANCHOR:]
    geom_rows.append({
        "seed": int(seed),
        "rank": int(np.linalg.matrix_rank(A)),
        "kappa_2": float(np.linalg.cond(A)),
        "anchor_row_norm_min": float(row_norms[:N_ANCHOR].min()),
        "anchor_row_norm_median": float(np.median(row_norms[:N_ANCHOR])),
        "anchor_row_norm_max": float(row_norms[:N_ANCHOR].max()),
        "cloud_row_norm_min": float(row_norms[N_ANCHOR:].min()),
        "cloud_row_norm_median": float(np.median(row_norms[N_ANCHOR:])),
        "cloud_row_norm_max": float(row_norms[N_ANCHOR:].max()),
        "anchor_lev_min": float(anchor.min()),
        "anchor_lev_mean": float(anchor.mean()),
        "anchor_lev_median": float(np.median(anchor)),
        "anchor_lev_max": float(anchor.max()),
        "cloud_lev_min": float(cloud.min()),
        "cloud_lev_mean": float(cloud.mean()),
        "cloud_lev_median": float(np.median(cloud)),
        "cloud_lev_max": float(cloud.max()),
        "mean_anchor_cloud_lev_ratio": float(anchor.mean() / cloud.mean()),
        "median_anchor_cloud_lev_ratio": float(np.median(anchor) / np.median(cloud)),
        "sum_leverage": float(lev.sum()),
    })

reproduced_diag = pd.DataFrame(geom_rows)
reproduced_diag.to_csv(REPRO_DIR / "A2_matrix_diagnostics.csv", index=False)

released_diag = pd.read_csv(DATA_DIR / "A2_matrix_diagnostics.csv")
shared_cols = list(released_diag.columns)
check = released_diag.merge(reproduced_diag[shared_cols], on="seed", suffixes=("_saved", "_new"))
for col in shared_cols:
    if col == "seed":
        continue
    a = check[f"{col}_saved"].to_numpy()
    b = check[f"{col}_new"].to_numpy()
    assert np.allclose(a, b, rtol=1e-12, atol=1e-14), col

print("All saved geometry diagnostics reproduce numerically.")
print("Condition number range:",
      f"{reproduced_diag['kappa_2'].min():.6f} -- {reproduced_diag['kappa_2'].max():.6f}")
print("Median across matrices, planned mean-based ratio:",
      f"{reproduced_diag['mean_anchor_cloud_lev_ratio'].median():.10f}")
print("Median across matrices, reported median-based ratio:",
      f"{reproduced_diag['median_anchor_cloud_lev_ratio'].median():.10f}")


## 3. Lewis iteration

For $r=1-2/p$, we factor $W^{r/2}A=QR$, solve $R^\top X^\top=A^\top$, and set

$$
w\_i^+=w\_i^r\|X\_{i,:}\|\_2^2.
$$

Each run starts from $w\_i^{(0)}=0.2$. We record the first strict crossings of $10^{-4}$, $10^{-6}$, and $10^{-8}$.


In [ ]:
def lewis_update_rescaled_qr(A, w, p):
    r = 1.0 - 2.0 / p
    B = (w ** (r / 2.0))[:, None] * A
    _, R = np.linalg.qr(B, mode="reduced")
    X = np.linalg.solve(R.T, A.T).T
    return (w ** r) * np.sum(X * X, axis=1)


def run_lewis(A, p, max_iter=MAX_ITER):
    m, n = A.shape
    w = np.full(m, n / m, dtype=np.float64)
    crossing = {tau: None for tau in THRESHOLDS}
    residuals = []

    for update_count in range(1, max_iter + 1):
        w_new = lewis_update_rescaled_qr(A, w, p)
        R = float(np.max(np.abs(np.log(w_new / w))))
        residuals.append(R)

        for tau in THRESHOLDS:
            if crossing[tau] is None and R < tau:
                crossing[tau] = update_count

        w = w_new
        if R < PRIMARY_TOL:
            break

    return {
        "p": float(p),
        "T_1e-4": crossing[1e-4],
        "T_1e-6": crossing[1e-6],
        "T_1e-8": crossing[1e-8],
        "reached_primary_tol": crossing[1e-8] is not None,
        "updates_run": update_count,
        "final_residual": residuals[-1],
        "min_final_w": float(w.min()),
        "max_final_w": float(w.max()),
        "final_sum_w": float(w.sum()),
        "sum_w_error": float(abs(w.sum() - n)),
    }, np.asarray(residuals)


## 4. Fresh rerun

This cell runs all 750 cases and writes the fresh run summaries and residual trajectories to `reproduced/`.


In [ ]:
run_rows = []
trajectory_rows = []
start = time.perf_counter()

for matrix_id, (seed, A) in enumerate(zip(FINAL_SEEDS, regenerated_matrices), start=1):
    kappa = float(reproduced_diag.loc[reproduced_diag["seed"] == seed, "kappa_2"].iloc[0])
    for p in P_GRID:
        t0 = time.perf_counter()
        summary, residuals = run_lewis(A, float(p))
        summary_row = {
            "matrix_id": matrix_id,
            "seed": int(seed),
            "kappa_2": kappa,
            **summary,
            "runtime_sec": time.perf_counter() - t0,
        }
        run_rows.append(summary_row)
        trajectory_rows.extend({
            "matrix_id": matrix_id,
            "seed": int(seed),
            "p": float(p),
            "update_count": k,
            "residual": float(R),
        } for k, R in enumerate(residuals, start=1))

reproduced_run = pd.DataFrame(run_rows)
reproduced_traj = pd.DataFrame(trajectory_rows)

reproduced_run.to_csv(REPRO_DIR / "A2_run_summary.csv", index=False)
reproduced_traj.to_csv(REPRO_DIR / "A2_residual_trajectories.csv", index=False)

assert len(reproduced_run) == 750
assert reproduced_run[["matrix_id", "p"]].duplicated().sum() == 0
assert reproduced_run["reached_primary_tol"].all()
assert reproduced_run["updates_run"].max() <= MAX_ITER

print("Fresh runs completed:", len(reproduced_run))
print("All reached 1e-8:", bool(reproduced_run["reached_primary_tol"].all()))
print("Maximum iterations:", int(reproduced_run["updates_run"].max()))
print("Fresh-run wall time (s):", round(time.perf_counter() - start, 2))


## 5. Recompute summaries and compare with the released results

Runtime is environment-dependent and is not compared. We compare the matrix/$p$ cases, threshold-crossing counts, and the final summaries.


In [ ]:
def make_primary_summary(run_df):
    return (
        run_df.groupby("p", sort=True)["T_1e-8"]
        .agg(
            n="count",
            q20=lambda x: x.quantile(0.20),
            median="median",
            q80=lambda x: x.quantile(0.80),
            min="min",
            max="max",
        )
        .reset_index()
    )


def make_tolerance_summary(run_df):
    return (
        run_df.groupby("p", sort=True)[["T_1e-4", "T_1e-6", "T_1e-8"]]
        .median()
        .reset_index()
    )

reproduced_primary = make_primary_summary(reproduced_run)
reproduced_tolerance = make_tolerance_summary(reproduced_run)
reproduced_primary.to_csv(REPRO_DIR / "A2_primary_summary.csv", index=False)
reproduced_tolerance.to_csv(REPRO_DIR / "A2_tolerance_summary.csv", index=False)

released_run = pd.read_csv(DATA_DIR / "A2_run_summary.csv")
released_primary = pd.read_csv(DATA_DIR / "A2_primary_summary.csv")
released_tolerance = pd.read_csv(DATA_DIR / "A2_tolerance_summary.csv")

keys = ["matrix_id", "seed", "p"]
count_cols = ["T_1e-4", "T_1e-6", "T_1e-8", "updates_run"]
cmp = released_run[keys + count_cols].merge(
    reproduced_run[keys + count_cols], on=keys, suffixes=("_saved", "_new"), validate="one_to_one"
)

count_diffs = []
for col in count_cols:
    d = np.abs(cmp[f"{col}_saved"].to_numpy() - cmp[f"{col}_new"].to_numpy())
    count_diffs.append(d)
    print(f"{col}: exact matches {np.sum(d == 0)}/750; max |difference| = {int(d.max())}")

primary_diff = np.max(np.abs(
    released_primary[["q20", "median", "q80", "min", "max"]].to_numpy(float)
    - reproduced_primary[["q20", "median", "q80", "min", "max"]].to_numpy(float)
))

tol_diff = np.max(np.abs(
    released_tolerance[["T_1e-4", "T_1e-6", "T_1e-8"]].to_numpy(float)
    - reproduced_tolerance[["T_1e-4", "T_1e-6", "T_1e-8"]].to_numpy(float)
))

# Exact agreement is expected in the retained environment. A small count-level
# difference can occur on another floating-point stack without changing the experiment.
max_count_diff = max(int(d.max()) for d in count_diffs)
if max_count_diff > 2 or primary_diff > 2 or tol_diff > 2:
    raise AssertionError(
        "Fresh rerun differs materially from the released iteration-count results. "
        "Inspect the numerical environment before using the reproduction."
    )

print("Maximum primary-summary difference (iterations):", primary_diff)
print("Maximum tolerance-summary difference (iterations):", tol_diff)
print("Scientific reproduction check: PASS")


## 6. Reconstruct the released residual crossing times directly

The released residual trajectories are enough to reconstruct all three saved first-crossing counts without rerunning the numerical iteration.


In [ ]:
released_traj = pd.read_csv(DATA_DIR / "A2_residual_trajectories.csv")
for tau, col in [(1e-4, "T_1e-4"), (1e-6, "T_1e-6"), (1e-8, "T_1e-8")]:
    crossings = (
        released_traj.loc[released_traj["residual"] < tau]
        .groupby(keys)["update_count"].min().rename("reconstructed").reset_index()
    )
    z = released_run[keys + [col]].merge(crossings, on=keys, validate="one_to_one")
    assert np.array_equal(z[col].to_numpy(), z["reconstructed"].to_numpy())

print("All 2,250 released threshold-crossing counts reconstruct exactly from the saved trajectories.")


## 7. A1 comparison and paper figure

The paper compares A2 with the final released A1 summary. The A1 and A2 matrix banks are different; this is a comparison of family-level summaries, not a pairing of individual matrices.

The plot below uses the same quantities as the released paper figure. The fresh figure is written to `reproduced/`.


In [ ]:
if not A1_PRIMARY_PATH.exists():
    raise FileNotFoundError(
        "The combined A1/A2 figure expects the released sibling file "
        "../A1/data/A1_primary_summary.csv"
    )

a1 = pd.read_csv(A1_PRIMARY_PATH)
a2 = reproduced_primary.copy()
diag = reproduced_diag.copy()

assert np.allclose(a1["p"], a2["p"], rtol=0, atol=1e-12)

comparison = a1[["p", "q20", "median", "q80"]].rename(columns={
    "q20": "A1_q20", "median": "A1_median", "q80": "A1_q80"
}).merge(
    a2[["p", "q20", "median", "q80"]].rename(columns={
        "q20": "A2_q20", "median": "A2_median", "q80": "A2_q80"
    }),
    on="p", validate="one_to_one"
)

comparison["A2_pct_more"] = 100 * (comparison["A2_median"] / comparison["A1_median"] - 1)
subset = comparison.loc[comparison["p"] >= 7.3, "A2_pct_more"]
print("A2 median increase for p >= 7.3:", f"{subset.min():.3f}% -- {subset.max():.3f}%")

ratio_vals = diag["median_anchor_cloud_lev_ratio"].to_numpy()
kappa_vals = diag["kappa_2"].to_numpy()
ratio_med = np.median(ratio_vals)
kappa_med = np.median(kappa_vals)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(7.4, 2.7))
p = comparison["p"].to_numpy()

ax0.plot(p, comparison["A1_median"], marker="o", markersize=3.5, linewidth=1.3, label="Gaussian")
ax0.fill_between(p, comparison["A1_q20"], comparison["A1_q80"], alpha=0.15)
ax0.plot(p, comparison["A2_median"], marker="s", markersize=3.5, linewidth=1.3,
         label=r"Structured ($\beta=1/4$)")
ax0.fill_between(p, comparison["A2_q20"], comparison["A2_q80"], alpha=0.15)
ax0.set_xlabel(r"$p$")
ax0.set_ylabel(r"Iterations to reach $R_t < 10^{-8}$")
ax0.grid(alpha=0.25)
ax0.legend(fontsize=8, loc="upper left", frameon=True)
ax0.text(0.02, 0.98, "(a)", transform=ax0.transAxes, ha="left", va="top", fontsize=10)

ax1.scatter(ratio_vals, kappa_vals, s=22, alpha=0.85)
ax1.scatter([ratio_med], [kappa_med], s=65, marker="D", label="Median", zorder=3)
ax1.axvline(ratio_med, linewidth=1.0, alpha=0.5)
ax1.axhline(kappa_med, linewidth=1.0, alpha=0.5)
ax1.set_xlim(min(15.0, ratio_vals.min() - 0.35), max(18.0, ratio_vals.max() + 0.35))
ax1.set_ylim(1.0, max(1.25, kappa_vals.max() + 0.03))
ax1.set_xlabel("Median anchor/cloud leverage ratio")
ax1.set_ylabel(r"$\kappa_2(A)$")
ax1.grid(alpha=0.25)
ax1.legend(fontsize=8, loc="upper right", frameon=True)
ax1.text(0.02, 0.98, "(b)", transform=ax1.transAxes, ha="left", va="top", fontsize=10)
ax1.text(
    0.03, 0.08,
    rf"ratio median $= {ratio_med:.2f}$" "\n"
    rf"$\kappa_2(A)$ median $= {kappa_med:.3f}$" "\n"
    rf"$\kappa_2(A)$ range $= [{kappa_vals.min():.3f},\, {kappa_vals.max():.3f}]$",
    transform=ax1.transAxes, ha="left", va="bottom", fontsize=8,
    bbox=dict(boxstyle="round,pad=0.25", alpha=0.12),
)

fig.tight_layout(w_pad=1.4)
fig.savefig(REPRO_DIR / "A2_two_panel_scaling_plus_geometry_compact.pdf", bbox_inches="tight")
fig.savefig(REPRO_DIR / "A2_two_panel_scaling_plus_geometry_compact.png", dpi=220, bbox_inches="tight")
plt.show()

print("Released paper figure: figures/A2_two_panel_scaling_plus_geometry_compact.pdf")
print("Fresh figure: reproduced/A2_two_panel_scaling_plus_geometry_compact.pdf")
print("The data used in both panels reproduce from the released A2 files.")


## 8. Final checks

The public files in `data/` and `figures/` are inputs to this notebook and are never overwritten. The `reproduced/` directory can be deleted after checking the rerun.


In [ ]:
expected_cases = set((int(seed), float(p)) for seed in FINAL_SEEDS for p in P_GRID)
observed_cases = set(zip(reproduced_run["seed"].astype(int), reproduced_run["p"].astype(float)))
assert observed_cases == expected_cases
assert len(reproduced_run) == 750
assert reproduced_run["reached_primary_tol"].all()
assert (reproduced_diag["rank"] == 20).all()
assert reproduced_diag["kappa_2"].max() <= np.sqrt(6) + 1e-12

print("PASS: 50 matrices x 15 p-values = 750 fresh runs.")
print("PASS: all fresh runs reached the primary threshold.")
print("PASS: matrix geometry matches the recorded construction.")
print("PASS: released data/ and figures/ were read only; fresh files are under reproduced/.")
